# Logistic Regression + TF-IDF

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

## Load Data

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DACN/data/data.csv")

print(df.head())

                      content label  start
0               Áo bao đẹp ạ!   POS      5
1                   Tuyệt vời   POS      5
2   2day ao khong giong trong   NEG      1
3  Mùi thơm,bôi lên da mềm da   POS      5
4            Vải đẹp, dày dặn   POS      5


In [ ]:
# Mapping label
label_map = {
    "NEG": 0,
    "NEU": 1,
    "POS": 2
}

df["label"] = df["label"].map(label_map)

# Chỉ giữ 2 cột cần thiết
df = df[["content", "label"]]

# Xoá dòng bị NaN nếu có
df = df.dropna()

print(df.head())

                      content  label
0               Áo bao đẹp ạ!      2
1                   Tuyệt vời      2
2   2day ao khong giong trong      0
3  Mùi thơm,bôi lên da mềm da      2
4            Vải đẹp, dày dặn      2


## EDA

In [ ]:
from collections import Counter
import re
texts = df["content"].astype(str)

# tách từ
words = []
for text in texts:
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    words.extend(tokens)

# lọc từ có độ dài 1 hoặc 2
short_words = [w for w in words if len(w) <= 2]

# đếm tần suất
counter = Counter(short_words)

# sắp xếp nhiều -> ít
result = counter.most_common(30)

for word, freq in result:
    print(word, freq)

áo 4421
và 4011
vụ 2529
ko 1938
có 1828
k 1788
ok 1683
mà 1591
là 1398
bị 1236
sẽ 1110
ạ 1053
1 1013
hộ 1012
sp 810
đc 779
so 724
y 716
ý 688
2 659
đã 648
dễ 584
rẻ 563
ơn 509
m 507
cả 502
ra 475
5 471
ng 450
ổn 445


In [ ]:
def normalize_text(text):
    text = text.lower()

    # thay từ lóng phổ biến
    slang_dict = {
        "khong": "không",
        "ko": "không",
        "k": "không",
        "hok": "không",
        "giong": "giống",
        "sp": "sản phẩm",
        "mik": "mình",
        "dc": "được",
        "đc": "được"
    }

    for k, v in slang_dict.items():
        text = text.replace(k, v)

    # bỏ ký tự lặp nhiều lần (đẹpppp → đẹp)
    text = re.sub(r'(.)\1+', r'\1', text)

    # bỏ ký tự đặc biệt
    text = re.sub(r'[^\w\s]', ' ', text)

    # chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["content"] = df["content"].apply(normalize_text)

## Chia dữ liệu

In [ ]:
X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## TF_IDF

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.85,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model.predict(X_test_tfidf)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print("Accuracy:", accuracy)
print("F1-score:", f1)

print(classification_report(y_test, y_pred))

Accuracy: 0.7854643765903307
F1-score: 0.759006066317798
              precision    recall  f1-score   support

           0       0.72      0.77      0.74      1353
           1       0.45      0.19      0.27       987
           2       0.84      0.94      0.89      3948

    accuracy                           0.79      6288
   macro avg       0.67      0.63      0.63      6288
weighted avg       0.75      0.79      0.76      6288



In [ ]:
def predict(text):
    text = normalize_text(text)

    vec = vectorizer.transform([text])

    pred = model.predict(vec)[0]

    label_map = {
        0: "NEG",
        1: "NEU",
        2: "POS"
    }

    return label_map[pred]

In [ ]:
print(predict("Sản phẩm này rất tốt"))
print(predict("Chất lượng quá tệ"))
print(predict("sp k quá tốt cũng k quá tệ"))
print(predict("bth"))
print(predict("Sp này mik cảm thấy thật sự chưa tuyệt vời lắm"))
print(predict("Sp này mik cảm thấy chưa tệ"))
print(predict("Sp này mik cảm thấy k thật sự tuyệt"))

POS
NEG
NEG
NEU
POS
NEG
NEG


In [ ]:
print(predict("Sản phẩm này rất tốt"))
print(predict("không giống ảnh "))
print(predict("mắc rất khó chịu"))
print(predict("tạm chấp nhận đc nhưng k ấn tượng lắm"))
print(predict("woah"))
print(predict("sương sương"))
print(predict("quá tệ"))
print(predict("woah, k thể tin nỗi là sp quá tệ"))
print(predict("Sp này mik cảm thấy k nên mua"))
print(predict("cảm giác thật sự chưa tuyệt"))
print(predict("vải đẹp, da mềm có hương thơm nhma khi ship đến thì mẫu k giống shop đã đưa lên"))

POS
NEG
NEG
NEU
POS
POS
NEG
NEG
NEG
POS
POS


In [ ]:
# import joblib

# joblib.dump(model, "/content/drive/MyDrive/DACN/LogisticRegresion/best_model/lr_model.pkl")
# joblib.dump(vectorizer, "/content/drive/MyDrive/DACN/LogisticRegresion/best_model/tfidf_vectorizer.pkl")

In [ ]:
!nvidia-smi

Sat Mar 14 16:20:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!cat /proc/meminfo | grep MemTotal

MemTotal:       13286948 kB


In [ ]:
import psutil
psutil.virtual_memory()

svmem(total=13605834752, available=11746500608, percent=13.7, used=1523691520, free=7871582208, active=695672832, inactive=4579995648, buffers=184745984, cached=4025815040, shared=2170880, slab=281686016)

In [ ]:
!python --version

Python 3.12.12
